In [1]:
# ========== 导入：周末练习「技术问答」所需工具 ==========

# load_dotenv：把 .env 里的密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# Markdown / display / update_display：展示与「流式刷新」同一块 Markdown
from IPython.display import Markdown, display, update_display
# OpenAI 客户端：这里会指向本地 Ollama 的 /v1 兼容接口
from openai import OpenAI
# ollama 官方 Python 包：本笔记本导入了，后续单元格主要走 OpenAI 兼容路径
import ollama


In [2]:
# ========== 常量：本地 Llama 模型名 ==========
# 字符串必须与本机 ollama pull / ollama list 中的名字一致
MODEL_LLAMA = 'llama3.2'


In [10]:
# ========== 环境 + 客户端：连到本机 Ollama（OpenAI 兼容） ==========

# 设置环境：加载 .env（即使本地占位 key，也保持与课程习惯一致）
load_dotenv()
# base_url 指向 Ollama 的 /v1；api_key 本地常填任意非空占位 "ollama"
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


In [11]:
# ========== 提问内容：一段要解释的 Python 代码 ==========
# 三引号多行字符串；发给模型时保持英文原样（改译会改变题目）
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


In [12]:
# ========== 组装 system / user 两段 prompt 文本 ==========
# system：设定「技术导师」人设与覆盖领域（字符串保持英文）
system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"
# user：在固定前缀后拼上具体 question
user_prompt = "Please give a detailed explanation to the following question: "+question


In [13]:
# ========== messages 列表：Chat Completions 的标准输入结构 ==========
messages = [
    # system 在前：定角色与回答风格
    {"role":"system","content": system_prompt},
    # user 在后：真正的问题
    {"role":"user", "content":user_prompt}
]


In [41]:
# ========== 发起流式请求：stream=True 返回可迭代 chunk ==========
# 注意：拿到的是 stream 迭代器；若下一格按非流式去取 .message，可能不符合预期
stream = openai.chat.completions.create(model=MODEL_LLAMA, messages=messages,stream=True)


In [40]:
# ========== 探索格：按「非流式响应」方式取 content ==========
# 教学提示：上一格 stream=True 时，通常应 for chunk in stream；本格写法更像非流式对象
# 保留原逻辑不动，便于对照你本地跑出来的报错/现象
display(Markdown(stream.choices[0].message.content))


**Explanation of the given code**

The code snippet you provided appears to be written in Python. It utilizes the `dict` data type that includes a subset called `items()` which allows us to iterate over both the key-value pairs and the dictionary as a whole (`dict.items()` instead of just iterating) for accessing their members.

Here is how you can break it down:

1. `{book.get("author") for book in books if book.get("author")}`

   This part of the code creates an iterator (`generator expression`) that produces values from each dictionary that contain `author` as a key.

   Here's what's happening in more detail:
   - `for book in books`: Iterate over each item (dictionary) in the list called `books`.
   - `if book.get("author")`: Skip to the next dictionary when its value for `"author"` is not defined. This prevents any potential KeyError.
   - `book.get("author")`: Return the author's name from each dictionary that has a valid "author".

2. `yield from`

   When used inside an existing generator or iterator, `yield from` will take values from another generator iterates over them one after another and yield each of the yielded values themselves.

So when combining these with a dict like this:

```python
books = [
    {"id": 1, "title": "Python for Data Science", "author": "John Smith"},
    {"id": 2, "title": "Data Analysis with Python", "author": None},
    {"id": 3, "title": "Introduction to Machine Learning", "author": "Emily Davis"}
]

for author in yield from {book.get("author") for book in books if book.get("author")}:
    print(author)
```

It will allow us to get all the authors while skipping any that don't exist.

In [42]:
# ========== 正确的流式消费：边收 delta 边 update_display ==========

# 累积完整回答文本
response = ""
# 先占位显示空 Markdown，拿到 display_id 以便原地刷新（不必每次新开一块输出）
display_handle = display(Markdown(""), display_id=True)
# 逐块迭代 stream；每个 chunk 里 choices[0].delta.content 可能是一小段或 None
for chunk in stream:
    # 拼接增量；None 当成空串，避免 TypeError
    response += chunk.choices[0].delta.content or ''
    # 去掉代码围栏噪音，方便纯 Markdown 展示
    response = response.replace("```","").replace("markdown", "")
    # 调试草稿（保持注释）
    # print(chunk.choices[0].delta.content 或 '')
    # 用同一 display_id 刷新内容，实现「打字机」效果
    update_display(Markdown(response), display_id=display_handle.display_id)


Let's break down this line of code.

**What is a ` yielding` function?**

In Python, `yield` is a keyword used to define generators. A generator is similar to a list comprehension, but instead of returning a new list, it returns an iterable that generates its values on the fly when iterated. This allows you to create functions that can produce multiple values over time, without having to store them all at once in memory.

**The `yield from` expression**

Now, let's talk about `yield from`. It was introduced in Python 3.3 as a way to delegate iteration to another iterator. When used with the `yield` keyword, it tells Python to yield values one by one from the input sequence.

In essence, `yield from` is like saying: "instead of yielding all my values at once, use the values of this other iterable and yield each one in order". This allows us to build complex iterators by combining smaller ones.

**The code**

Now that we have a basic understanding of generators and `yield`, let's get back to our original line of code:

python
yield from {book.get("author") for book in books if book.get("author")}


Here, the inner dictionary comprehension is generating values. The outer expression uses `yield from` to delegate iteration to this inner iterable.

Let's break down how it works:

1. `{book.get("author") for book in books if bookget("author")}`: This is a dictionary comprehension that creates an inner iterable. It does the following:
    - For each `book` in `books`, it calls `.get("author")` and includes its value in the resulting dictionary.
    - The `if book.get("author")` part is a filter on the input data, allowing only books with authors to pass through into the inner iterable.

2. However, this result of the inner comprehension is **not iterable** by itself (it returns an empty dictionary). This would cause a TypeError when trying to iterate over it using `yield from`.

To fix this, we wrap the inner comprehension in parentheses, as `{}` will create an empty dictionary and return values if `.get("author")` isn't None. Then the values are iterated using `yield from`, generating each value one by one.

Here's how you might write this more idiomatically:

python
def get_authors():
    for book in books:
        yield book.get('author') or None

# Usage: 
for author in get_authors():
    print(author)


In summary, the line `yield from {book.get("author") for book in books if book.get("author")}` is using generators to:

- Create an inner iterable with an iterative method by calling `.get()` on objects
- Use `yield from` to create another generator that yields these iterables' values.

**Example Use Cases:**

This technique can be applied when you want to generate all unique keys or elements (iterables) in multiple lists, dictionaries, or any other iterable with similar properties, while avoiding memory issues like those seen with regular for loops.

In [60]:
# ========== 封装：根据用户问题构造 messages ==========

def set_user_prompt(prompt):
    # 调试草稿（保持注释不执行）
    # print("我的提示"+提示)
    # system prompt 保持英文：影响模型行为
    system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"
    # 这里的 user_prompt 直接等于传入的整段用户问题
    user_prompt = prompt
    # 拼成 OpenAI messages 结构
    messages = [
        {"role":"system","content": system_prompt},
        {"role":"user", "content":user_prompt}
    ]
    # 末尾分号是作者原写法，Python 允许，保持不动
    return messages;


In [61]:
# ========== 封装：构造 messages 并创建流式 completion ==========

def call_stream_openai(user_prompt):
    # 先把用户问题变成 messages
    messages = set_user_prompt(user_prompt)
    # 再对流式 API；返回 stream 迭代器给上层消费
    stream = openai.chat.completions.create(model=MODEL_LLAMA, messages=messages,stream=True)
    return stream


In [62]:
# ========== 封装：端到端流式聊天并刷新 Markdown ==========

def my_chat(user_prompt):
    # 简单日志：确认函数被调用
    print("my chat called")
    # 拿到流式 stream
    stream = call_stream_openai(user_prompt)
    # 累积文本
    response = ""
    # 创建可更新的 Markdown 显示句柄
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # 拼增量 token/文本块
        response += chunk.choices[0].delta.content or ''
        # 清洗围栏标记
        response = response.replace("```","").replace("markdown", "")
        # 原地刷新展示
        update_display(Markdown(response), display_id=display_handle.display_id)


In [70]:
# ========== 交互入口：终端/笔记本输入问题，再流式作答 ==========
# input 会阻塞等待你输入；在 Jupyter 里会弹出输入框
my_question = input("Please enter your question:")
# 把问题交给 my_chat，边生成边刷新 Markdown
my_chat(my_question)


Please enter your question: textarea in python input


my chat called


**Creating a Text Area Input in Python**

To create a text area input in Python, you can use the `ttk.Entry` widget from the Tkinter library. Here is an example of how to do it:
python
import tkinter as tk
from tkinter import ttk

# Create the main window
root = tk.Tk()
root.title("Text Area Input")

# Create a text area widget
textarea = ttk.Entry(root, width=50, height=10)

# Add a label and the input field to the window
label = tk.Label(root, text="Enter some text:")
label.pack()
textarea.pack()

# Function to handle button click
def submit_text():
    text = textarea.get("1.0", "end-1c")
    print(text)

# Create a submit button
submit_button = tk.Button(root, text="Submit", command=submit_text)
submit_button.pack()

# Start the Tkinter event loop
root.mainloop()

This code creates a simple window with a label and a text area input field. When you click the "Submit" button, it prints the text you entered in the text area to the console.

**Example Use Cases**

* Creating a simple form or survey where users can enter their responses.
* Allowing users to input and edit data in a web-based application.
* Capturing user input from a command-line interface (CLI) application.

**Tips and Variations**

* You can adjust the `width` and `height` parameters of the `ttk.Entry` widget to change the size of the text area input field.
* You can add other widgets or features to the window, such as buttons, radio buttons, checkboxes, or a dropdown menu, by adding more instances of `tk.Label`, `tk.Button`, etc.